In [10]:
import pandas as pd
import urllib.request
import yfinance as yf

In [11]:
# ── Step 1: fetch S&P 500 list from iShares IVV (updated daily) ────────────
import requests
from io import StringIO

# ── Step 1: fetch S&P 500 list from Wikipedia using requests ───────────────
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
})

response = session.get(url)
print(f"Status code: {response.status_code}")

sp500 = pd.read_html(StringIO(response.text))[0]
sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500 = sp500.reset_index(drop=True)

print(f"Total tickers: {len(sp500)}")
print(sp500["GICS Sector"].value_counts())
print(sp500[["Symbol", "GICS Sector"]].head(10))

Status code: 200
Total tickers: 503
GICS Sector
Industrials               81
Financials                76
Information Technology    74
Health Care               59
Consumer Discretionary    47
Consumer Staples          34
Utilities                 31
Real Estate               31
Materials                 26
Communication Services    23
Energy                    21
Name: count, dtype: int64
  Symbol             GICS Sector
0    MMM             Industrials
1    AOS             Industrials
2    ABT             Health Care
3   ABBV             Health Care
4    ACN  Information Technology
5   ADBE  Information Technology
6    AMD  Information Technology
7    AES               Utilities
8    AFL              Financials
9      A             Health Care


In [12]:
# ── Step 2: how many stocks to pick per sector ─────────────────────────────
sector_targets = {
    "Information Technology": 12,
    "Health Care":            10,
    "Financials":             10,
    "Consumer Discretionary":  8,
    "Industrials":             8,
    "Communication Services":  7,
    "Consumer Staples":        7,
    "Energy":                  6,
    "Utilities":               5,
    "Real Estate":             4,
    "Materials":               3,
}  # sums to 80

In [13]:
test = yf.download("AAPL", start="2015-01-01", end="2024-12-31",
                   auto_adjust=True, progress=False)
print("Columns:", test.columns.tolist())
print("Shape:", test.shape)
print(test.head(2))

Columns: [('Close', 'AAPL'), ('High', 'AAPL'), ('Low', 'AAPL'), ('Open', 'AAPL'), ('Volume', 'AAPL')]
Shape: (2515, 5)
Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2015-01-02  24.192600  24.659502  23.754464  24.648438  212818400
2015-01-05  23.511053  24.042127  23.325178  23.962466  257142000


In [14]:
# ── Step 3: get average volume for every S&P 500 ticker ───────────────────
# This takes ~10-15 minutes. Runs once, then we screen from the results.

def get_avg_volume(ticker, start="2015-01-01", end="2024-12-31", min_trading_days=2000):
    """
    Returns average daily volume if ticker passes screening, else None.
    Screens for: sufficient history, liquidity, data quality.
    """
    try:
        df = yf.download(ticker, start=start, end=end,
                         auto_adjust=True, progress=False)

        if df is None or df.empty:
            return None

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        if len(df) < min_trading_days:
            return None
        if df["Close"].isnull().mean() > 0.01:
            return None

        avg_vol = float(df["Volume"].mean())
        if avg_vol < 500_000:
            return None

        return avg_vol

    except Exception as e:
        print(f"  ERROR {ticker}: {e}")
        return None

In [15]:
# ── Step 4: pick top-N by volume per sector ────────────────────────────────
print("\nFetching volume data... (~10-15 mins)\n")

volume_data = []
total = len(sp500)

for i in range(total):
    ticker = sp500.loc[i, "Symbol"]
    sector = sp500.loc[i, "GICS Sector"]
    avg_vol = get_avg_volume(ticker)

    status = f"{avg_vol:,.0f}" if avg_vol is not None else "FAILED"
    print(f"[{i+1:3d}/{total}] {ticker:<10} {sector:<35} {status}")

    volume_data.append({
        "ticker": ticker,
        "sector": sector,
        "avg_volume": avg_vol
    })



Fetching volume data... (~10-15 mins)

[  1/503] MMM        Industrials                         3,554,169
[  2/503] AOS        Industrials                         1,248,308
[  3/503] ABT        Health Care                         6,133,019
[  4/503] ABBV       Health Care                         7,325,530
[  5/503] ACN        Information Technology              2,284,802
[  6/503] ADBE       Information Technology              2,945,269
[  7/503] AMD        Information Technology              57,930,142
[  8/503] AES        Utilities                           6,180,263
[  9/503] AFL        Financials                          3,350,498
[ 10/503] A          Health Care                         1,998,473
[ 11/503] APD        Materials                           1,223,680
[ 12/503] ABNB       Consumer Discretionary              FAILED
[ 13/503] AKAM       Information Technology              1,775,267
[ 14/503] ALB        Materials                           1,638,077
[ 15/503] ARE        Rea


1 Failed download:
['FDXF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1420088400, endDate = 1735621200")')


[196/503] FDXF       Industrials                         FAILED
[197/503] FIS        Financials                          3,117,605
[198/503] FITB       Financials                          6,230,432
[199/503] FSLR       Information Technology              2,232,469
[200/503] FE         Utilities                           4,131,636
[201/503] FISV       Financials                          3,083,329
[202/503] FLEX       Information Technology              5,829,422
[203/503] F          Consumer Discretionary              52,362,147
[204/503] FTNT       Information Technology              7,721,790
[205/503] FTV        Industrials                         3,105,828
[206/503] FOXA       Communication Services              FAILED
[207/503] FOX        Communication Services              FAILED
[208/503] BEN        Financials                          3,261,974
[209/503] FCX        Materials                           21,900,401
[210/503] GRMN       Consumer Discretionary              1,037,201
[2


1 Failed download:
['HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1420088400, endDate = 1735621200")')


[236/503] HONA       Industrials                         FAILED
[237/503] HON        Industrials                         3,259,324
[238/503] HRL        Consumer Staples                    2,392,155
[239/503] HST        Real Estate                         8,014,783
[240/503] HWM        Industrials                         3,880,995
[241/503] HPQ        Information Technology              12,110,739
[242/503] HUBB       Industrials                         FAILED
[243/503] HUM        Health Care                         1,216,764
[244/503] HBAN       Financials                          12,365,401
[245/503] HII        Industrials                         FAILED
[246/503] IBM        Information Technology              4,797,254
[247/503] IEX        Industrials                         FAILED
[248/503] IDXX       Health Care                         552,236
[249/503] ITW        Industrials                         1,299,597
[250/503] INCY       Health Care                         1,659,205
[251/50


1 Failed download:
['Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1420088400, endDate = 1735621200")')


[386/503] Q          Information Technology              FAILED
[387/503] RL         Consumer Discretionary              1,096,514
[388/503] RJF        Financials                          1,232,329
[389/503] RTX        Industrials                         6,626,761
[390/503] O          Real Estate                         3,135,509
[391/503] REG        Real Estate                         987,160
[392/503] REGN       Health Care                         798,467
[393/503] RF         Financials                          12,042,360
[394/503] RSG        Industrials                         1,283,546
[395/503] RMD        Health Care                         798,097
[396/503] RVTY       Health Care                         768,939
[397/503] HOOD       Financials                          FAILED
[398/503] ROK        Industrials                         883,168
[399/503] ROL        Industrials                         1,531,562
[400/503] ROP        Information Technology              FAILED
[401/503] ROS


1 Failed download:
['SNDK']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01 -> 2024-12-31) (Yahoo error = "Data doesn\'t exist for startDate = 1420088400, endDate = 1735621200")')


[405/503] SNDK       Information Technology              FAILED
[406/503] SBAC       Real Estate                         865,262
[407/503] SLB        Energy                              10,515,026
[408/503] STX        Information Technology              3,516,489
[409/503] SRE        Utilities                           2,980,224
[410/503] NOW        Information Technology              8,242,346
[411/503] SHW        Materials                           1,814,425
[412/503] SPG        Real Estate                         2,137,458
[413/503] SWKS       Information Technology              2,368,468
[414/503] SJM        Consumer Staples                    1,008,929
[415/503] SW         Materials                           FAILED
[416/503] SNA        Industrials                         FAILED
[417/503] SOLV       Health Care                         FAILED
[418/503] SO         Utilities                           4,861,479
[419/503] LUV        Industrials                         7,300,843
[420/503

In [16]:
# ── Step 5: build dataframe ────────────────────────────────────────────────
vol_df = pd.DataFrame(volume_data)
passed = vol_df.dropna(subset=["avg_volume"])
print(f"\n✓ {len(passed)} tickers passed out of {total}")
print(passed["sector"].value_counts())



✓ 440 tickers passed out of 503
sector
Financials                69
Industrials               62
Information Technology    62
Health Care               52
Consumer Discretionary    42
Consumer Staples          32
Utilities                 30
Real Estate               28
Materials                 23
Communication Services    21
Energy                    19
Name: count, dtype: int64


In [17]:
# ── Step 6: pick top-N by volume per sector ────────────────────────────────
selected = []
print("\n" + "=" * 60)
print(f"{'FINAL SELECTION':^60}")
print("=" * 60)

for sector, target_n in sector_targets.items():
    sector_df = passed[passed["sector"] == sector].sort_values(
        "avg_volume", ascending=False
    )
    picked = sector_df.head(target_n)

    print(f"\n{sector} — {len(picked)}/{target_n} picked")
    print(f"  {'Ticker':<12} {'Avg Daily Volume':>20}")
    print(f"  {'-'*12} {'-'*20}")
    for _, r in picked.iterrows():
        print(f"  {r['ticker']:<12} {r['avg_volume']:>20,.0f}")

    selected.append(picked)

selected_df = pd.concat(selected).reset_index(drop=True)


                      FINAL SELECTION                       

Information Technology — 12/12 picked
  Ticker           Avg Daily Volume
  ------------ --------------------
  NVDA                  467,755,662
  AAPL                  117,116,177
  AMD                    57,930,142
  INTC                   33,110,233
  MSFT                   28,885,595
  AVGO                   28,345,113
  MU                     25,346,386
  CSCO                   22,081,205
  LRCX                   19,407,299
  SMCI                   14,406,509
  KLAC                   13,727,187
  HPE                    13,121,233

Health Care — 10/10 picked
  Ticker           Avg Daily Volume
  ------------ --------------------
  PFE                    29,001,401
  BMY                    10,863,425
  MRK                    10,600,807
  GILD                    8,755,684
  BSX                     8,177,313
  JNJ                     7,721,086
  VTRS                    7,687,591
  CVS                     7,660,981
  ABBV 

In [18]:
# ── Step 7: final output ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"TOTAL SELECTED: {len(selected_df)} stocks")
print("=" * 60)

final_tickers = selected_df["ticker"].tolist()
print(f"\nCopy this into config.py as SP500_EQUITIES:")
print(final_tickers)


TOTAL SELECTED: 80 stocks

Copy this into config.py as SP500_EQUITIES:
['NVDA', 'AAPL', 'AMD', 'INTC', 'MSFT', 'AVGO', 'MU', 'CSCO', 'LRCX', 'SMCI', 'KLAC', 'HPE', 'PFE', 'BMY', 'MRK', 'GILD', 'BSX', 'JNJ', 'VTRS', 'CVS', 'ABBV', 'ABT', 'BAC', 'WFC', 'C', 'JPM', 'HBAN', 'RF', 'KEY', 'PYPL', 'XYZ', 'MS', 'TSLA', 'AMZN', 'F', 'CMG', 'CCL', 'GM', 'NCLH', 'BKNG', 'CSX', 'GE', 'DAL', 'UAL', 'FAST', 'BA', 'LUV', 'RTX', 'NFLX', 'T', 'GOOGL', 'GOOG', 'META', 'CMCSA', 'VZ', 'WMT', 'KO', 'MO', 'KR', 'PG', 'MDLZ', 'KHC', 'XOM', 'KMI', 'OXY', 'HAL', 'SLB', 'CVX', 'PCG', 'NEE', 'EXC', 'AES', 'PPL', 'HST', 'KIM', 'WY', 'DOC', 'FCX', 'NEM', 'MOS']


In [ ]:
#Test compute_reutn.py function
from fetch_universe import fetch_prices
from compute_return import compute_all  # matches your filename

# load cached prices — no re-download
prices = fetch_prices()
ret, mu, Sigma = compute_all(prices)

print(f"Returns shape:       {ret.shape}")
print(f"Mu shape:            {mu.shape}")
print(f"Sigma shape:         {Sigma.shape}")

print(f"\nTop 5 by expected return:")
print(mu.nlargest(5).apply(lambda x: f"{x:.2%}"))

print(f"\nBottom 5 by expected return:")
print(mu.nsmallest(5).apply(lambda x: f"{x:.2%}"))

print(f"\nMost volatile assets:")
vol = pd.Series(np.sqrt(np.diag(Sigma.values)), index=Sigma.columns)
print(vol.nlargest(5).apply(lambda x: f"{x:.2%}"))

print(f"\nAny NaNs in returns? {ret.isnull().any().any()}")
print(f"Any NaNs in mu?      {mu.isnull().any()}")
print(f"Any NaNs in Sigma?   {Sigma.isnull().any().any()}")

Loading prices from cache...
Loaded 100 assets, 1726 days from cache
Returns shape:       (1725, 100)
Mu shape:            (100,)
Sigma shape:         (100, 100)

Top 5 by expected return:
SOL-USD    77.46%
BNB-USD    57.51%
ETH-USD    44.61%
NVDA       44.47%
SMCI       38.85%
dtype: object

Bottom 5 by expected return:
INTC    -13.71%
TLT      -7.45%
NEM      -4.12%
PYPL     -3.13%
CVS      -2.55%
dtype: object

Most volatile assets:
SOL-USD    106.87%
BNB-USD     70.72%
ETH-USD     65.75%
SMCI        62.32%
NCLH        57.05%
dtype: object

Any NaNs in returns? False
Any NaNs in mu?      False
Any NaNs in Sigma?   False
